# Hai nguồn OCR để **ĐO**: PaddleOCR và VietOCR → Volume `keyframes-btc`

`docs/triet/ke-hoach-offline-ingest.md` §2.2 · so với `docs/triet/handoff.md` §5.3, §6.4

### Vì sao có notebook này

`n5_ocr_qwen` đo được **10,8 giây/ảnh** khi gọi từng ảnh — hơn 2 giờ cho một video
307 keyframe, và ~532 giờ GPU cho 177.321 keyframe của batch 1. Gộp lô hạ được con
số đó nhưng không đổi bậc độ lớn. Câu hỏi là: **có bộ đọc nào rẻ hơn mà không tệ
hơn không.**

Notebook này chạy hai bộ đọc nữa trên **cùng một video**, ghi ra hai artifact
riêng, rồi đặt cả bốn cạnh nhau.

| tầng | artifact | trên Volume | phát hiện vùng chữ | nhận dạng chữ |
|---|---|---|---|---|
| `n5_ocr_paddle` | `paddle_ocr` | `parts/paddle_ocr/` | PaddleOCR | PaddleOCR (`lang='vi'`) |
| `n5_ocr_vietocr` | `viet_ocr` | `parts/viet_ocr/` | **PaddleOCR** | VietOCR |

**VietOCR chỉ nhận dạng, không phát hiện.** Đưa cả khung hình vào nó thì nó vẫn
trả về một chuỗi — không ném lỗi, chỉ ra rác. Nên nó bắt buộc phải ghép với một bộ
phát hiện, và ở đây bộ đó là của PaddleOCR, **cùng tham số** với tầng trên. Hai
tầng cũng xếp hộp chữ bằng cùng `group_text_lines`. Nhờ vậy chênh lệch giữa hai
file là chênh lệch của **bộ nhận dạng tiếng Việt**, không phải của bộ phát hiện —
đúng cái lỗi so-sai-đối-tượng mà `handoff.md` §8.2 ghi lại.

### Ba thứ notebook này CỐ Ý không làm

1. **Không đụng vào `n5_ocr_easyocr` hay `n5_ocr_qwen`.** Hai artifact cũ giữ
   nguyên. `handoff.md` §5.3 đo được arm `ocr` (easyocr) đóng góp **+0,089**, gấp 4
   lần arm Qwen — "đừng đụng vào easyocr" là kết luận có số đo đứng sau.
2. **Không thêm hai tầng này vào `config/ingest_shards_batch1_btc_images.yaml`.**
   Thêm vào là `merge_bundle.py` sẽ đòi đủ mảnh `paddle_ocr` cho **cả 873 video**
   và từ chối cả bundle. Danh sách tầng ở đây được khai thẳng trong notebook.
3. **Không ghi `part-manifest`.** Hai artifact này chưa thuộc hợp đồng đầu ra của
   bundle; ghi vào manifest là tuyên bố ngược lại. Marker vẫn được cắm (để chạy
   tiếp được khi phiên đứt), nhưng dưới tên `probe` — xem ô Step 0c.

### Chưa có arm nào đọc chúng

Cả hai artifact đăng ký với `source=None`: không có `source` Elasticsearch, không
có arm, không vào `ingest_sota_text.py`. Chúng được sinh ra để **so**, và việc cắm
vào tầng online là quyết định sau khi nhìn số — không phải bây giờ.

## Step 0a — công tắc, repo, gói nền

In [ ]:
VIDEO       = "L21_V001"        # <-- video để đo. Đổi được, nhưng đo 1 video là đủ.
SHARDS      = "config/ingest_shards_batch1_btc_images.yaml"
TRANSPORT   = "modal"
VOLUME      = "keyframes-btc"   # CÙNG volume với notebook 02/03
VOL_VERSION = 2

# Danh sách tầng KHAI THẲNG Ở ĐÂY, không đọc từ `plan.stages` — xem ô mô tả trên.
# Đưa hai tầng này vào yaml sẽ khiến `merge_bundle.py` đòi đủ mảnh cho 873 video.
PROBE_STAGES = ["n5_ocr_paddle", "n5_ocr_vietocr"]

# Chạy lại DÙ marker đã báo xong. Cần khi đổi tham số; nó GHI ĐÈ part + marker cũ.
REDO_STAGES  = []

REPO_URL = "https://github.com/AIVIETNAM-AIO-Triet-Descartes/multimodal-fusion-engine.git"
REPO_REF = "feature/triet/structure"

import os, subprocess, sys
from pathlib import Path

REPO = Path("/kaggle/input/mfe-repo")
if not REPO.is_dir():
    REPO = Path("/kaggle/working/mfe")
    if not REPO.is_dir():
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        url = REPO_URL.replace("https://", f"https://{token}@")
        subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_REF, url, str(REPO)],
                       check=True, capture_output=True)   # token không lọt vào log
sys.path.insert(0, str(REPO / "src"))

req = REPO / "notebooks" / "requirements-kaggle.lock.txt"
if not req.is_file():
    req = REPO / "notebooks" / "requirements-kaggle-btc-images.txt"
print(f"môi trường nền: {req.name}")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=True)

## Step 0b — PaddleOCR và VietOCR

> ⚠️ **Ô này đã từng làm hỏng `torch` của cả phiên.** Triệu chứng, đo được
> 19-08-2026: `ImportError: libtorch_cuda.so: undefined symbol: ncclCommShrink`
> ném ra ở Step 2, chứ không ở đây.
>
> Nguyên nhân là bản cũ cài `paddlepaddle-gpu` bằng `-i <index của Paddle>`. Cờ
> `-i` **thay thế** PyPI chứ không thêm vào, nên pip giải luôn *toàn bộ phụ thuộc*
> từ index đó — trong đó có các gói `nvidia-*`. Nó hạ `nvidia-nccl-cu12` xuống bản
> Paddle cần, rồi bánh xe `paddlepaddle-gpu` **thất bại** (nên `DEVICE` ra `cpu`).
> Kết quả: gói GPU không cài được, nhưng NCCL thì đã bị hạ — và `libtorch_cuda.so`
> của torch cần `ncclCommShrink`, một symbol chỉ có từ NCCL 2.27.
>
> Hai chỗ sửa: `--no-deps` để lần cài đó không được đụng vào gói nào khác, và
> `TRY_PADDLE_GPU = False` làm mặc định.

**Mặc định giờ là GPU, và đó là một lần sửa lại.** Bản trước để CPU cho an toàn,
lập luận là "chậm hơn nhưng vẫn đọc ra đúng chữ đó". Số đo bác bỏ: 307 ảnh chạy
**hơn 25 phút mà chưa xong**, vì CPU cộng với oneDNN đã tắt (để né lỗi PIR) là hai
lần chậm chồng lên nhau. Một phép đo không chạy xong thì không phải phép đo.

`--no-deps` mới là thứ làm việc bật GPU an toàn — nó chặn lần cài chạm vào các gói
`nvidia-*` của torch, đúng cái đã phá torch lần trước. Cài hỏng thì ô tự lùi về CPU
và nói rõ; lúc đó hãy giảm `LIMIT` ở Step 2 thay vì ngồi đợi.

**`vietocr` được cài kèm ràng buộc (`-c`).** Không có nó, pip được phép kéo torch
sang phiên bản khác, và tiền xử lý ảnh của SigLIP đổi theo — kiểu lệch **không ném
lỗi**, nó chỉ làm vector của bundle thành hỗn hợp hai phép đo (§2.5). Ràng buộc
gồm cả các gói `nvidia-*`, vì đó chính là thứ đã vỡ.

Ô cuối **kiểm lại torch ngay tại đây**. Một môi trường hỏng phải lộ ra ở ô cài
đặt, không phải hai ô sau dưới dạng một `ImportError` không ai đọc ra.

In [ ]:
from importlib.metadata import PackageNotFoundError, version

# BẬT MẶC ĐỊNH từ 19-08-2026, sau khi đo: Paddle trên CPU với oneDNN đã tắt chạy
# hơn 25 phút mà chưa xong một video 307 ảnh. `--no-deps` bên dưới là thứ làm việc
# bật này an toàn — nó chặn lần cài chạm vào `nvidia-*` của torch, đúng cái đã phá
# torch lần trước. Cài hỏng thì ô này tự lùi về CPU và nói rõ.
TRY_PADDLE_GPU   = True
PADDLE_GPU_WHEEL = "paddlepaddle-gpu==3.0.0"
PADDLE_GPU_INDEX = "https://www.paddlepaddle.org.cn/packages/stable/cu126/"

# Ràng buộc: gói mà đổi phiên bản là đổi SỐ ĐO, cộng với các gói CUDA mà ABI của
# torch phụ thuộc vào. `nvidia-*` có mặt ở đây vì chính chúng là thứ đã bị hạ.
pins = []
for package in ("torch", "torchvision", "transformers", "numpy", "pillow",
                "nvidia-nccl-cu12", "nvidia-cublas-cu12", "nvidia-cudnn-cu12",
                "nvidia-cuda-runtime-cu12"):
    try:
        pins.append(f"{package}=={version(package)}")
    except PackageNotFoundError:
        continue
constraints = Path("/kaggle/working/probe-constraints.txt")
constraints.write_text("\n".join(pins) + "\n", encoding="utf-8")
print("rang buoc:")
for pin in pins:
    print(f"  {pin}")

if TRY_PADDLE_GPU:
    # `--no-deps` KHONG phai toi uu: no la thu chan lan cai nay cham vao
    # `nvidia-*` cua torch. Thieu phu thuoc that thi `import paddle` se bao.
    ok = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--no-deps",
         PADDLE_GPU_WHEEL, "-i", PADDLE_GPU_INDEX],
    ).returncode == 0
    if not ok:
        print("KHONG cai duoc paddlepaddle-gpu — se dung ban CPU")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "-c", str(constraints),
     "-r", str(REPO / "notebooks" / "requirements-kaggle-btc-ocr-probe.txt")],
    check=True,
)

In [ ]:
# KIEM MOI TRUONG — phai chay truoc Step 2, khong phai de bao hiem.
# torch bi hong o day trong nhu mot loi cua tang OCR o Step 2, va no khong phai.
problems = []

# Đọc TRƯỚC `import paddle`: paddle đọc FLAGS lúc nạp. Đây là lớp đỡ thứ hai cho
# lỗi oneDNN dưới đây; lớp thứ nhất là `enable_mkldnn=False` mà tầng truyền vào
# hàm dựng. Hai lớp vì lớp thứ nhất phụ thuộc bản PaddleOCR có nhận tham số đó.
#
#     (Unimplemented) ConvertPirAttribute2RuntimeAttribute not support
#     [pir::ArrayAttribute<pir::DoubleAttribute>]  (… onednn_instruction.cc:116)
#
# Đo được 19-08-2026 trên Kaggle CPU, paddlepaddle 3.x: đường thực thi PIR mới của
# Paddle gặp một attribute mà nhánh oneDNN của nó chưa cài đặt. Không phải lỗi của
# ta và không sửa được từ đây — tắt oneDNN thì cả nhánh đó biến mất.
os.environ.setdefault("FLAGS_use_mkldnn", "0")

try:
    import torch
    print(f"torch {torch.__version__} · cuda={torch.cuda.is_available()}")
except Exception as err:
    problems.append(f"torch KHONG import duoc: {err}")

try:
    import paddle
    DEVICE = "gpu" if paddle.device.is_compiled_with_cuda() else "cpu"
    print(f"paddle {paddle.__version__} -> device={DEVICE}")
except Exception as err:
    DEVICE = "cpu"
    problems.append(f"paddle KHONG import duoc: {err}")

for package in ("paddleocr", "vietocr"):
    try:
        print(f"{package} {version(package)}")
    except PackageNotFoundError:
        problems.append(f"{package} chua cai")

if problems:
    print()
    for line in problems:
        print(f"  HONG: {line}")
    print()
    print("Neu la 'undefined symbol: ncclCommShrink' hoac tuong tu, mot lan cai da")
    print("ha cac goi nvidia-* xuong duoi ban torch can. Goi da cai VAN CON sau khi")
    print("restart kernel, nen phai chua:")
    print("    pip install -U nvidia-nccl-cu12")
    print("roi Restart session. Chac an nhat la 'Factory reset' roi chay lai tu dau")
    print("voi TRY_PADDLE_GPU = False.")
    raise SystemExit("moi truong chua san sang — dung o day, dung chay Step 2")

print()
print("moi truong OK")

## Step 0c — Modal, phân công, việc đã xong

`MEMBER = "probe"` chứ không phải tên người, và đó là có chủ ý. Marker đi vào
`/_status/probe/…` nên nó **không lọt vào bảng đếm của ai cả**: `verify_bundle.py`
chỉ duyệt các thành viên có trong `plan.members`, và `StageStatus` của Trọng/Quang
không thấy nó. Marker vẫn làm đúng việc của nó — phiên đứt giữa hai tầng thì lần
chạy sau đi tiếp đúng tầng còn lại.

Video vẫn phải thuộc phân công thật, vì ảnh lấy từ gói zip của người sở hữu nó.

In [ ]:
import logging, yaml
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

from kaggle_secrets import UserSecretsClient
from mfe.offline.bundle import BundleLayout, BundleWriter, StageStatus, load_shard_plan
from mfe.offline.stages import STAGES, StageContext, read_keyframes, unload_models
from mfe.offline.transport import build_transport

secrets = UserSecretsClient()
os.environ["MODAL_TOKEN_ID"] = secrets.get_secret("MODAL_TOKEN_ID")
os.environ["MODAL_TOKEN_SECRET"] = secrets.get_secret("MODAL_TOKEN_SECRET")

MEMBER    = "probe"            # KHÔNG phải tên người — xem ô mô tả trên
RAW       = yaml.safe_load((REPO / SHARDS).read_text(encoding="utf-8"))
plan      = load_shard_plan(REPO / SHARDS)
layout    = BundleLayout(plan.bundle)
transport = build_transport(TRANSPORT, volume=VOLUME,
                            local_root="/kaggle/working/bundle_local",
                            version=VOL_VERSION)
WORK = Path("/kaggle/working/scratch"); WORK.mkdir(parents=True, exist_ok=True)
ZIPS = Path("/kaggle/working/zips");    ZIPS.mkdir(parents=True, exist_ok=True)
KAGGLE_INPUT = Path("/kaggle/input")
writer = BundleWriter(transport, layout, MEMBER, WORK)

EXTRAS = RAW["extras"]
owner  = plan.owner_of(VIDEO)
if owner is None:
    raise SystemExit(f"{VIDEO} khong nam trong phan cong nao cua {SHARDS}")
MY_ZIP = next(name for name in plan.zips[owner] if VIDEO in plan.videos_in(name))

status = StageStatus.load(transport, layout, MEMBER)
todo   = [s for s in PROBE_STAGES if s in REDO_STAGES or not status.is_done(s, VIDEO)]

print(f"bundle : {plan.bundle}  ->  volume {VOLUME} (v{VOL_VERSION})")
print(f"video  : {VIDEO}  (goi {MY_ZIP}, thuoc phan cong cua {owner})")
print(f"tang   : {', '.join(PROBE_STAGES)}")
print(f"con lam: {', '.join(todo) if todo else '(khong con gi)'}")

# Điều kiện tiên quyết: notebook 02 phải nạp `map-keyframes` của video này lên
# Volume trước, nếu không thì không có gì để đối chiếu `frame_idx`.
prior = StageStatus.load(transport, layout, owner)
if not prior.is_done("n0_import_btc", VIDEO):
    print()
    print(f"CHU Y: {VIDEO} chua duoc nhap boi notebook 02 ({owner}). Chay 02 truoc.")

## Step 1 — ảnh của một video

Cùng đường lấy ảnh như notebook 03: ưu tiên Kaggle Dataset đã attach (đã giải nén
hoặc còn ZIP nguyên bản), thiếu cả hai mới tải. Chỉ bung **một** video.

`--read-timeout=60` là thứ quan trọng nhất trong lệnh `wget`: không có nó, một kết
nối bị bóp xuống vài trăm B/s vẫn là kết nối "sống" và wget đợi vô hạn.

In [ ]:
import shutil, zipfile

def validate_zip(path):
    try:
        with zipfile.ZipFile(path) as archive:
            if not archive.infolist():
                raise RuntimeError(f"{path}: ZIP rong")
    except zipfile.BadZipFile as err:
        raise RuntimeError(f"{path}: khong phai ZIP hoan chinh") from err


def find_kaggle_file(name, expect_bytes=None):
    """ZIP nguyên bản trong Kaggle Dataset đã attach. Read-only: không copy."""
    if not KAGGLE_INPUT.is_dir():
        return None
    found = sorted(p for p in KAGGLE_INPUT.rglob(name) if p.is_file())
    if not found:
        return None
    if len(found) != 1:
        raise RuntimeError(f"{name}: thay nhieu ban: {', '.join(str(p) for p in found)}")
    actual = found[0].stat().st_size
    if expect_bytes is not None and actual != expect_bytes:
        raise RuntimeError(f"{found[0]}: {actual:,} byte, cho doi {expect_bytes:,}")
    validate_zip(found[0])
    print(f"    cache Kaggle: {found[0]} ({actual / 2**30:.2f} GB)")
    return found[0]


def find_kaggle_extracted(archive_name, child):
    """Thư mục Kaggle đã tự giải nén sẵn từ một archive."""
    stem = Path(archive_name).stem
    if not KAGGLE_INPUT.is_dir():
        return None
    for root, directories, _files in os.walk(KAGGLE_INPUT):
        if stem in directories:
            extracted = Path(root) / stem / child
            if not extracted.is_dir():
                raise RuntimeError(f"{Path(root) / stem}: thieu thu muc {child}")
            print(f"    cache Kaggle da giai nen: {extracted}")
            return extracted
        directories[:] = [d for d in directories if d != stem]
    return None


def download(url, name, expect_bytes=None):
    cached = find_kaggle_file(name, expect_bytes)
    if cached is not None:
        return cached
    target = ZIPS / name
    if target.is_file() and expect_bytes is not None and target.stat().st_size == expect_bytes:
        validate_zip(target)
        print(f"    {name}: da co")
        return target
    print(f"    tai {name} …")
    subprocess.run(
        ["wget", "-q", "--show-progress", "-c",
         "--timeout=30", "--read-timeout=60", "--tries=0", "--waitretry=10",
         "--retry-connrefused", "-O", str(target), url],
        check=True,
    )
    if expect_bytes is not None and target.stat().st_size != expect_bytes:
        size = target.stat().st_size
        target.unlink(missing_ok=True)
        raise RuntimeError(f"{name}: tai ve {size:,} byte, cho doi {expect_bytes:,}. Da xoa.")
    validate_zip(target)
    return target


def extract_video(zip_path, video_id, into):
    out = into / video_id
    out.mkdir(parents=True, exist_ok=True)
    count = 0
    with zipfile.ZipFile(zip_path) as archive:
        for name in archive.namelist():
            parts = Path(name).parts
            if len(parts) < 2 or parts[-2] != video_id or name.endswith("/"):
                continue
            with archive.open(name) as src, (out / parts[-1]).open("wb") as dst:
                shutil.copyfileobj(src, dst, length=1 << 20)
            count += 1
    return out, count


def extract_one_file(zip_path, inner_suffix, dest):
    with zipfile.ZipFile(zip_path) as archive:
        match = [n for n in archive.namelist() if n.endswith(inner_suffix)]
        if not match:
            raise FileNotFoundError(f"{inner_suffix} khong co trong {zip_path.name}")
        dest.parent.mkdir(parents=True, exist_ok=True)
        with archive.open(match[0]) as src, dest.open("wb") as dst:
            shutil.copyfileobj(src, dst)
    return dest


print("map-keyframes:")
maps_dir = find_kaggle_extracted("map-keyframes-aic25-b1.zip", "map-keyframes")
if maps_dir is not None:
    MAP_CSV = maps_dir / f"{VIDEO}.csv"
    if not MAP_CSV.is_file():
        raise FileNotFoundError(f"{MAP_CSV}: khong ton tai")
else:
    maps_zip = download(EXTRAS["map_keyframes"], "map-keyframes-aic25-b1.zip")
    MAP_CSV = extract_one_file(maps_zip, f"/{VIDEO}.csv", WORK / "map-keyframes" / f"{VIDEO}.csv")

print(f"anh cua {VIDEO}:")
kf_root = find_kaggle_extracted(MY_ZIP, "keyframes")
if kf_root is not None:
    KF_DIR = kf_root / VIDEO
    if not KF_DIR.is_dir():
        raise FileNotFoundError(f"{VIDEO}: thieu trong {kf_root}")
    N_IMAGES = sum(1 for p in KF_DIR.iterdir() if p.is_file() and p.suffix.lower() == ".jpg")
else:
    kf_zip = download(plan.zip_urls[MY_ZIP], MY_ZIP, plan.zip_bytes.get(MY_ZIP))
    KF_DIR, N_IMAGES = extract_video(kf_zip, VIDEO, WORK / "keyframes")

KEYFRAMES = read_keyframes(MAP_CSV, KF_DIR)
print(f"    {N_IMAGES} anh · {len(KEYFRAMES)} dong trong map-keyframes")

## Step 2 — chạy hai tầng

`n5_ocr_paddle` chạy trước và nạp PaddleOCR vào cache; `n5_ocr_vietocr` **dùng lại
đúng đối tượng đó** (cùng khoá cache, cùng tham số), nên bộ phát hiện của hai tầng
là một, không phải hai bản giống nhau.

`n5_ocr_vietocr` chạy **hai pha**: Paddle phát hiện xong *toàn bộ* ảnh trước, rồi
VietOCR mới được nạp và đọc. Không phải để cho gọn — paddle và torch mỗi bên mang
oneDNN/OpenMP riêng, và xen kẽ chúng trong một vòng lặp là khác biệt duy nhất giữa
tầng này với `n5_ocr_paddle` vốn chạy trót lọt. Marker tách `detect_seconds` và
`recognize_seconds` để bạn đọc được nửa nào tốn thời gian.

Cái giá đã biết: tầng VietOCR chạy lại cả pipeline PaddleOCR rồi vứt phần chữ Paddle
đọc được, chỉ giữ hộp. Đổi lại là **một** đường mã thay vì hai đường "chỉ phát hiện"
khác nhau giữa PaddleOCR 2.x và 3.x — và phần bị vứt là phần rẻ, đo được bằng chính
`elapsed_s` của hai tầng in ra dưới đây.

> ⚠️ **Khởi tạo PaddleOCR hỏng một lần là phải Restart session.** PaddleOCR 3.x
> khởi tạo PaddleX ở phạm vi tiến trình, nên lần dựng thứ hai — kể cả khi lần đầu
> đã ném lỗi — chết bằng `RuntimeError: PDX has already been initialized.
> Reinitialization is not supported.` Chạy lại ô này sau một lần hỏng sẽ cho bạn
> **thông báo lỗi đó thay vì lỗi thật**. Đọc lỗi ĐẦU TIÊN, rồi restart.

In [ ]:
import time, traceback

# ⚠️ BA THỨ QUYẾT ĐỊNH LẦN CHẠY NÀY MẤT 3 PHÚT HAY 50 PHÚT.
#
# LIMIT — số ảnh mỗi tầng chạy; 0 = tất cả. Đo được 19-08: 307 ảnh trên CPU chạy
#   hơn 25 phút và vẫn chưa xong. Chất lượng đọc chữ của một bộ OCR đọc được từ
#   vài chục ảnh, nên chạy 40 trước, nhìn Step 3, rồi mới bỏ giới hạn.
#
# enable_mkldnn — oneDNN là ĐƯỜNG TĂNG TỐC CPU của Paddle. Tắt nó là đánh đổi
#   2–4× tốc độ để né lỗi `ConvertPirAttribute2RuntimeAttribute`. Cần thiết trên
#   CPU, nhưng nó biến "chậm" thành "rất chậm". Trên GPU, oneDNN không liên quan.
#
# progress_every — cứ N ảnh in một dòng tiến độ + ETA. Lần chạy đầu không in gì
#   suốt 25 phút nên từ ngoài không phân biệt được với treo.
LIMIT = 40          # <-- 0 = chạy hết. Bắt đầu bằng 40.

PADDLE_COMMON = {
    "lang": "vi",
    "device": DEVICE,
    "enable_mkldnn": False,     # bật lại được nếu DEVICE == "gpu"
    "limit": LIMIT,
    "progress_every": 10,
}

PARAMS = {
    "n5_ocr_paddle": {
        **PADDLE_COMMON,
        # 0.0 = giữ tất cả, giống easyocr `detail=0`. Nâng lên nếu muốn thấy
        # ảnh hưởng của việc lọc chữ đoán bừa — nó được ghi vào marker.
        "rec_score_min": 0.0,
    },
    "n5_ocr_vietocr": {
        **PADDLE_COMMON,   # PHẢI giống hệt, nếu không hai tầng dựng hai Paddle khác nhau
        "vietocr_config": "vgg_transformer",   # 'vgg_seq2seq' nhẹ hơn, kém hơn
        "vietocr_device": "cuda:0" if DEVICE == "gpu" else "cpu",
        "beamsearch": False,
        "crop_pad": 2,
    },
}

ctx = StageContext(VIDEO, Path("/dev/null"), KEYFRAMES, writer, layout, WORK)
markers = {}

for stage in PROBE_STAGES:
    if stage not in REDO_STAGES and status.is_done(stage, VIDEO):
        print(f"{stage:<16} bo qua (marker da bao xong)")
        continue
    print(f"{stage:<16} chay tren {LIMIT or len(KEYFRAMES)}/{len(KEYFRAMES)} keyframe …")
    started = time.perf_counter()
    try:
        ctx.params = dict(PARAMS.get(stage, {}))
        marker = STAGES[stage](ctx)
        markers[stage] = marker
        status.done.add((stage, VIDEO))
        print(f"{'':<16} {marker['rows']:>4}/{marker['images']} khoa co chu · "
              f"{marker['boxes']:>5} hop · {marker['elapsed_s']:.0f}s "
              f"({marker['seconds_per_image']:.2f}s/anh)")
    except Exception as err:
        print(f"[LOI] {stage}: {err}")
        traceback.print_exc(limit=4)
    finally:
        print(f"{'':<16} tong {time.perf_counter() - started:.0f}s")

# KHÔNG ghi `part-manifest`: hai artifact này chưa thuộc hợp đồng đầu ra của
# bundle, và ghi vào manifest là tuyên bố ngược lại. Marker đã đủ để chạy tiếp.
unload_models()

## Step 3 — bốn nguồn OCR cạnh nhau

Kéo cả bốn artifact của **cùng video** về và đặt cạnh nhau. Đây là thứ notebook này
tồn tại để tạo ra.

Đọc bảng thế nào:

* **`khoa co chu`** cao không tự nó là tốt. easyocr ra chữ trên *cả* 307 ảnh của
  `L21_V001`, phần lớn là logo và đồng hồ đọc sai — `handoff.md` §5.3 đo được nó
  đọc đúng `HTV9` **0/153** lần.
* **`s/anh`** là con số quyết định có chạy nổi 873 video hay không. Cột `batch 1`
  ngoại suy thẳng từ nó lên 177.321 keyframe, và nó là **ước lượng** — một video,
  một phiên, một loại nội dung.
* Cột **`chi minh no co`** cho thấy nguồn nào bắt được thứ nguồn khác bỏ sót. Đó
  mới là thứ fusion ăn được: hai arm sai ở những chỗ khác nhau thì cộng lại có lợi,
  còn hai arm giống nhau thì thêm cái thứ hai không được gì (§8.6).

In [ ]:
import json, tempfile
from mfe.contracts.ids import build_keyframe_id

SOURCES = ["ocr_text", "sviptr_ocr", "paddle_ocr", "viet_ocr"]
LABEL   = {"ocr_text": "easyocr", "sviptr_ocr": "Qwen2-VL",
           "paddle_ocr": "PaddleOCR", "viet_ocr": "VietOCR"}
BATCH1_KEYFRAMES = 177_321   # docs/triet/handoff.md §0 — bo keyframe cua BTC

def fetch(artifact):
    with tempfile.TemporaryDirectory() as tmp:
        local = Path(tmp) / f"{artifact}.json"
        try:
            transport.get(layout.part(artifact, VIDEO), local)
        except Exception:
            return None
        return json.loads(local.read_text(encoding="utf-8"))

payloads = {name: fetch(name) for name in SOURCES}
keys = [build_keyframe_id(VIDEO, kf.frame_idx) for kf in KEYFRAMES]
present = [n for n in SOURCES if payloads[n] is not None]

print(f"{VIDEO} · {len(keys)} keyframe\n")
header = f"{'nguon':<11} {'khoa co chu':>11} {'s/anh':>7} {'batch 1':>9} {'chi minh no co':>15}"
print(header)
print("-" * len(header))
for name in SOURCES:
    body = payloads[name]
    if body is None:
        print(f"{LABEL[name]:<11} {'(chua chay)':>11}")
        continue
    others = [payloads[o] for o in present if o != name]
    only = sum(1 for k in keys
               if body.get(k, '').strip() and not any(o.get(k, '').strip() for o in others))
    marker = markers.get({'paddle_ocr': 'n5_ocr_paddle', 'viet_ocr': 'n5_ocr_vietocr'}.get(name))
    if marker:
        per = marker["seconds_per_image"]
        hours = per * BATCH1_KEYFRAMES / 3600
        speed, projected = f"{per:.2f}", f"{hours:,.0f}h"
    else:
        speed, projected = "-", "-"   # do o phien khac, khong bia lai
    print(f"{LABEL[name]:<11} {len(body):>11} {speed:>7} {projected:>9} {only:>15}")

print("\n`batch 1` la NGOAI SUY tu mot video, khong phai so do tren batch 1.")

In [ ]:
# Đọc tay vài keyframe. Không có phép đo tự động nào thay được bước này: cả bốn
# nguồn đều có thể trả về một chuỗi trông hợp lý mà sai — `handoff.md` §5.3 đo
# được easyocr đọc `HTV9` sai 153/153 lần mà không có gì báo.
SAMPLES = 8

# Ưu tiên những keyframe mà các nguồn BẤT ĐỒNG nhau nhất — chỗ đồng ý thì không
# học được gì, chỗ bất đồng mới nói nguồn nào đọc được cái gì.
def disagreement(key):
    texts = {(payloads[n].get(key, "").strip().lower()) for n in present}
    return len(texts)

for key in sorted(keys, key=lambda k: (-disagreement(k), k))[:SAMPLES]:
    print(f"\n=== {key} ===")
    for name in present:
        text = payloads[name].get(key, "").strip()
        shown = " / ".join(text.splitlines())[:110] if text else "(trong)"
        print(f"  {LABEL[name]:<10}: {shown}")

## Step 4 — sau khi nhìn số

Việc còn lại **không phải** của notebook này:

* Nếu một nguồn thắng rõ, nó vẫn cần một quyết định của Lead trước khi cắm vào tầng
  online — `handoff.md` §9.1: *"tuyệt đối không tự quyết định việc thay thế hay cắt
  bớt model/API"*. Cắm vào nghĩa là đặt `source` cho artifact trong
  `ARTIFACTS`, thêm khoá tương ứng vào `sota_payloads.SOURCE_FILES`, rồi khai một
  arm — ba chỗ, và assertion trong `bundle.py` sẽ bắt nếu thiếu chỗ nào.
* Số ở đây là **một video**. Trước khi kết luận về batch 1, chạy thêm vài video có
  loại nội dung khác (`L25_*` là bài giảng, không phải bản tin) — `L21_V001` toàn
  logo đài và tiêu đề bản tin, và đó là loại chữ dễ nhất trong bộ.
* So với file phẳng của Bảo theo từng keyframe:

```
python scripts/compare_ocr.py --video L21_V001 --artifact paddle_ocr \
    --sota "D:/Projects/SOTA_AIC/data/vectors/sviptr_ocr.json"
```